# Stage 4 — train Pup on a Colab GPU

**Before anything else:** *Runtime → Change runtime type → T4 GPU*, then
*Runtime → Run all*. Training runs unattended; go and do Stage 4 tasks 4 and 5
(`export.py` and `NumpyPolicy`) on your laptop while it does.

What this notebook does:

1. clones **your** fork and installs it with `uv`,
2. asserts a GPU is actually visible (a CPU Colab will "work" and take a day),
3. runs `pup.train.train_ppo`,
4. plots the learning curve, renders a rollout, exports the numpy policy,
5. zips everything for download.

Keep this tab open. Free-tier Colab reclaims idle sessions, and while
checkpoints are written at every evaluation, a reclaimed session still costs you
the run in progress. See `docs/04_training_with_brax.md`.

In [ ]:
# ---- edit these three, then Run all -------------------------------------
REPO_URL = "https://github.com/kc0817/HRC-onboarding-fall26.git"
CONFIG   = "full"      # "full" (200M steps) | "t4_fast" (30M, use if you are short on time)
SEED     = 0
# -------------------------------------------------------------------------

: 

In [ ]:
import os, subprocess, sys
from pathlib import Path

# MUJOCO_GL must be set BEFORE anything imports mujoco. egl = GPU offscreen.
os.environ["MUJOCO_GL"] = "egl"

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
if not Path("/content/pup-onboarding").exists():
    subprocess.run(["git", "clone", REPO_URL, "/content/pup-onboarding"], check=True)
os.chdir("/content/pup-onboarding")
subprocess.run(["uv", "sync", "--extra", "gpu", "--python", "3.11"], check=True)

In [ ]:
# If this cell fails you are on a CPU runtime: Runtime -> Change runtime type -> T4 GPU.
subprocess.run(["uv", "run", "python", "-c",
                "import jax; print(jax.devices()); "
                "assert jax.default_backend() == 'gpu', 'no GPU runtime'"], check=True)
subprocess.run(["uv", "run", "python", "scripts/check_setup.py"], check=False)

## Train

Expect tens of minutes. Each line of output is one evaluation: watch
`eval/episode_reward` climb while `eval/avg_episode_length` stays near 1000. If
episode length collapses, the robot is falling — see the Stage 4 doc's "reading
the curve" section.

This cell needs your Stage 4 task 1 (the wiring block in `train_ppo.py`) to be
done and pushed.

In [ ]:
subprocess.run(["uv", "run", "python", "-m", "pup.train.train_ppo",
                "--env", "PupJoystickFlat", "--config", CONFIG,
                "--seed", str(SEED), "--out", "runs/colab"], check=True)

# If the session dies part-way, re-run this cell with the last checkpoint:
#   ... "--out", "runs/colab", "--restore", "runs/colab/checkpoints/<step>"

In [ ]:
import csv
import matplotlib.pyplot as plt

Path("results").mkdir(exist_ok=True)
rows = list(csv.DictReader(open("runs/colab/learning_curve.csv")))
steps  = [float(r["steps"]) for r in rows if r.get("eval/episode_reward")]
reward = [float(r["eval/episode_reward"]) for r in rows if r.get("eval/episode_reward")]
length = [float(r["eval/avg_episode_length"]) for r in rows if r.get("eval/avg_episode_length")]

figure, (top, bottom) = plt.subplots(2, 1, sharex=True, figsize=(7, 6))
top.plot(steps, reward); top.set_ylabel("eval/episode_reward"); top.grid(alpha=0.3)
bottom.plot(steps, length, color="tab:orange")
bottom.set_ylabel("eval/avg_episode_length"); bottom.set_xlabel("environment steps")
bottom.grid(alpha=0.3)
figure.tight_layout()
figure.savefig("results/learning_curve.png", dpi=140)
plt.show()

## Evaluate, export, and download

`eval.json` is the file the reviewer reads. `walking_passes` must be `true` for
the acceptance criteria in `docs/04_training_with_brax.md`.

The export cell needs your Stage 4 task 4 (`export_policy`).

In [ ]:
import json
report = json.loads(Path("runs/colab/eval.json").read_text())
print(json.dumps(report, indent=2))
print("\nwalking_passes:", report["walking_passes"])

In [ ]:
subprocess.run(["uv", "run", "python", "-m", "pup.train.export",
                "--checkpoint", "runs/colab/policy.pkl",
                "--out", "results/pup_policy.npz"], check=True)

from IPython.display import Image, display
display(Image(filename="runs/colab/rollout.gif"))

In [ ]:
import shutil
shutil.copy("runs/colab/eval.json", "results/eval_training.json")
shutil.copy("runs/colab/rollout.gif", "results/training.gif")
shutil.make_archive("pup-training", "zip", "runs/colab")

from google.colab import files
files.download("pup-training.zip")
print("Commit results/learning_curve.png, results/eval_training.json, "
      "results/training.gif and results/pup_policy.npz to your repository.")

## Where this shows up in NEMO

This is the club's training loop, on Colab instead of the lab machine. NEMO runs
the same sequence — Playground environment, Brax PPO, tracking metrics, export,
sim2sim validation — and the habit of *reading `eval.json` before calling a run
successful* is the one worth taking with you. A rising reward curve with a
policy that cannot track a 1 m/s command is a very common way to fool
yourself.